# 34 - Humanoid Locomotion RL Intuition

## What / Why / How

**What we are trying to do:** Build a toy locomotion policy loop with rewards, domain randomization, and gait features.

**Why this matters:** Modern humanoid walking is often trained in simulation with reinforcement learning and transferred to hardware.

**How we will do it:** Optimize simple gait parameters under randomized dynamics and evaluate robustness instead of one perfect nominal run.

## Locomotion RL Toy

Humanoid-Gym, Isaac Lab, and related systems train locomotion in high-throughput physics. Here we only practice the idea: optimize gait parameters under randomized conditions.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(34)

def evaluate_gait(params, randomized=True):
    step_length, cadence, lateral_gain = params
    friction = rng.uniform(0.55, 1.2) if randomized else 0.9
    actuator = rng.uniform(0.8, 1.2) if randomized else 1.0
    terrain = rng.normal(0, 0.03) if randomized else 0.0
    speed = actuator * step_length * cadence
    target_speed = 0.8
    stability_penalty = max(0, abs(lateral_gain + terrain) - friction * 0.35) ** 2
    energy = 0.3 * cadence**2 + 0.8 * step_length**2
    reward = 2.0 - abs(speed - target_speed) - 3.0 * stability_penalty - 0.15 * energy
    return reward

population = rng.uniform([0.1, 1.0, -0.2], [0.55, 3.0, 0.2], size=(80, 3))
for generation in range(30):
    scores = np.array([np.mean([evaluate_gait(p) for _ in range(8)]) for p in population])
    elites = population[np.argsort(scores)[-10:]]
    population = elites.mean(axis=0) + rng.normal(0, elites.std(axis=0) + 1e-3, size=(80, 3))
    population = np.clip(population, [0.05, 0.5, -0.4], [0.7, 4.0, 0.4])
best = population[np.argmax([np.mean([evaluate_gait(p) for _ in range(20)]) for p in population])]
print("best gait [step_length, cadence, lateral_gain]:", best)

In [ ]:
robust_scores = [evaluate_gait(best, randomized=True) for _ in range(300)]
nominal_scores = [evaluate_gait(best, randomized=False) for _ in range(50)]
print("nominal mean reward:", np.mean(nominal_scores))
print("randomized mean reward:", np.mean(robust_scores))
print("randomized 10th percentile:", np.percentile(robust_scores, 10))

In [ ]:
phase = np.linspace(0, 2*np.pi, 120)
left_foot = best[0] * np.sin(phase)
right_foot = best[0] * np.sin(phase + np.pi)
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(phase, left_foot, label="left foot swing")
    plt.plot(phase, right_foot, label="right foot swing")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add a fall penalty when friction is low.
2. Add terrain slope as a randomized variable.
3. Explain why this toy optimizer is not enough for real walking.